In [0]:
Gen AI: simple app. 
(rag)
Agent: do action 
agentic ai: gmail (mcp)

agent: 
    LLM (brain) + Hands(Tools) 

In [0]:
%pip install -U --quiet \
    databricks-langchain \
    langchain \
    langchain-community \
    wikipedia \
    youtube_search \
    duckduckgo-search\
    -U ddgs \
    -U langgraph

dbutils.library.restartPython()

In [0]:
import mlflow

# Set experiment for better organization
mlflow.set_experiment("/Users/naval.datamaster@gmail.com/agent")

# Enable autologging BEFORE creating the LangChain client
mlflow.langchain.autolog()

In [0]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import YouTubeSearchTool
from langchain_community.tools import DuckDuckGoSearchRun

In [0]:
wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper()
)

result = wiki_tool.invoke("Generative AI?")

print(result)

In [0]:
youtube_tool = YouTubeSearchTool()

result = youtube_tool.invoke("Independence day 2026")

print(result)

In [0]:
search_tool = DuckDuckGoSearchRun()

result = search_tool.invoke("Latest Databricks AI features")

print(result)

In [0]:
tools = [
    wiki_tool,
    youtube_tool,
    search_tool
]

for tool in tools:
    print("Name:", tool.name)
    print("Description:", tool.description)
    print("-" * 50)

In [0]:
User
 ↓
LLM
 ↓
Decide which tool to use
 ↓
Tool
 ↓
Tool result
 ↓
LLM
 ↓
Final answer

In [0]:
User
 ↓
Movie Agent
 ↓
Wikipedia → movie information
 ↓
YouTube → trailer
 ↓
Web Search → latest information
 ↓
LLM
 ↓
Recommendation

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent


# LLM

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)

tools = [
    wiki_tool,
    youtube_tool,
    search_tool
]

# Agent
agent = create_agent(
    model=llm,
    tools=tools
)


# User query
query = """
Recommend a good science-fiction movie.

Use Wikipedia to find information about the movie
and YouTube to find its trailer.

Give me:
1. Movie name
2. Release year
3. Short description
4. Why you recommend it
5. YouTube trailer
"""


# Invoke agent
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})


# Final answer
print(result["messages"][-1].content)

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent


# -----------------------------------
# 1. Databricks Foundation Model
# -----------------------------------

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)


# -----------------------------------
# 2. Tools
# -----------------------------------

tools = [
    wiki_tool,
    youtube_tool,
    search_tool
]


# -----------------------------------
# 3. Create Agent
# -----------------------------------

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful movie research assistant.

Use:
- Wikipedia for factual movie information
- YouTube for movie trailers
- Web search for current information

Always use the appropriate tools to verify information.
"""
)


# -----------------------------------
# 4. Get Movie Name from User
# -----------------------------------

movie_name = input("🎬 Enter the movie name: ")

query = f"""
Tell me about the movie "{movie_name}".

Use Wikipedia to find information about the movie
and YouTube to find its trailer.

Provide:

1. Movie name
2. Release year
3. Director
4. Main actors
5. Short description
6. Why someone should watch it
7. YouTube trailer
"""


# -----------------------------------
# 5. Invoke Agent
# -----------------------------------

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})


# -----------------------------------
# 6. Display Final Answer
# -----------------------------------

print("\n🎬 Movie Information")
print("=" * 60)

print(result["messages"][-1].content)

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent


# -----------------------------------
# 1. Databricks Foundation Model
# -----------------------------------

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)


# -----------------------------------
# 2. Tools
# -----------------------------------

tools = [
    wiki_tool,
    youtube_tool,
    search_tool
]


# -----------------------------------
# 3. Create Agent
# -----------------------------------

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful movie research assistant.

Use:
- Wikipedia for factual movie information
- YouTube for movie trailers
- Web search for current information

Always use the appropriate tools to verify information.
"""
)


# -----------------------------------
# 4. Get Movie Name from User
# -----------------------------------

movie_name = input("🎬 Enter the movie name: ")

query = f"""
Tell me about the movie "{movie_name}".

Use Wikipedia to find information about the movie
and YouTube to find its trailer.

Provide:

1. Movie name
2. Release year
3. Director
4. Main actors
5. Short description
6. Why someone should watch it
7. YouTube trailer
"""


# -----------------------------------
# 5. Invoke Agent
# -----------------------------------

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})


# -----------------------------------
# 6. Display Final Answer
# -----------------------------------

print("\n🎬 Movie Information")
print("=" * 60)

print(result["messages"][-1].content)

In [0]:
system_prompt = """
You are a movie research assistant.

When the user provides a movie name, research and provide the following:

1. Movie name
2. IMDb rating
3. Summary of reviews and overall critical reception
4. OTT platforms where the movie is currently available
5. YouTube trailer link

Tool usage:
- Use Wikipedia for factual information and movie details.
- Use DuckDuckGo for IMDb rating, reviews, OTT availability, and current information.
- Use YouTube to find the official movie trailer.
- Do not make up information.
- If reliable information cannot be found, clearly say that it could not be verified.

Present the final answer in a clear numbered format.
"""

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent


# -----------------------------------
# 1. Databricks Foundation Model
# -----------------------------------

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)


# -----------------------------------
# 2. Tools
# -----------------------------------

tools = [
    wiki_tool,
    youtube_tool,
    search_tool
]

# -----------------------------------
# 2. system prompt
# -----------------------------------


system_prompt = """
You are a movie research assistant.

The user will provide a movie name. Your job is to identify the movie
the user most likely intends and then research it using the available tools.

Return:

1. Movie name
2. IMDb rating
3. Summary of reviews and overall critical reception
4. OTT platforms where the movie is currently available
5. YouTube trailer links

--------------------------------------------------
MOVIE IDENTIFICATION
--------------------------------------------------

Movie names may contain:
- spelling mistakes
- typos
- missing words
- extra spaces
- transliteration differences
- incomplete titles
- incorrect capitalization

If the movie name appears misspelled or ambiguous, DO NOT immediately
accept the first search result.

Use DuckDuckGo and Wikipedia to identify the most likely intended movie.

Compare information such as:
- movie title
- release year
- actors
- director
- language
- recent popularity
- search-result context

If multiple movies have the same or similar name, prefer the movie that
best matches the user's likely intent and the most relevant/current result.

For example:
"Dhurandhar" may refer to more than one movie. Search broadly and
determine the most relevant movie before providing the answer.

If there is a clear likely match, use that movie without asking the user
to correct the spelling.

If there are multiple equally likely matches, ask the user to clarify.

--------------------------------------------------
TOOL USAGE
--------------------------------------------------

Wikipedia:
Use for movie facts, cast, director, release information, and background.

DuckDuckGo:
Use for IMDb rating, reviews, current OTT availability, recent information,
and resolving ambiguous movie titles.

YouTube:
Use to find the movie's official trailer or the most relevant official
trailer.

For current information such as OTT availability, always prefer recent
search results.

--------------------------------------------------
VERIFICATION
--------------------------------------------------

Do not rely on a single search result.

Before giving the final answer, cross-check important information using
multiple sources whenever possible.

Do not invent information.

If a specific piece of information cannot be verified, clearly say:
"Information could not be verified."

Do not say that information is unavailable simply because the first tool
search failed. Try another relevant tool or search with alternative
queries first.

--------------------------------------------------
FINAL RESPONSE
--------------------------------------------------

Once the movie has been identified, provide the answer in this format:

1. Movie name:
2. IMDb rating:
3. Reviews:
4. OTT availability:
5. YouTube trailer:

Keep the response concise and useful.

If you corrected the user's movie title, mention the corrected title
briefly at the beginning.
"""
# -----------------------------------
# 3. Create Agent
# -----------------------------------

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)


# -----------------------------------
# 4. Get Movie Name from User
# -----------------------------------

movie_name = input("🎬 Enter the movie name: ")



# -----------------------------------
# 5. Invoke Agent
# -----------------------------------

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": movie_name
        }
    ]
})


# -----------------------------------
# 6. Display Final Answer
# -----------------------------------

print("\n🎬 Movie Information")
print("=" * 60)

print(result["messages"][-1].content)

In [0]:
CATALOG = "dev"
SCHEMA = "bronze"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.movie_research_agent"

In [0]:
import mlflow
from mlflow.pyfunc import PythonModel


class MovieAgentModel(PythonModel):

    def __init__(self, agent):
        self.agent = agent

    def predict(self, context, model_input):

        movie_name = model_input["movie_name"][0]

        result = self.agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": movie_name
                }
            ]
        })

        return result["messages"][-1].content

In [0]:
movie_model = MovieAgentModel(agent)

In [0]:
test_input = {
    "movie_name": ["Dhurandhar"]
}

In [0]:
result = movie_model.predict(
    None,
    test_input
)

print(result)

In [0]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

In [0]:
with mlflow.start_run() as run:

    model_info = mlflow.pyfunc.log_model(
        artifact_path="movie_agent",
        python_model=movie_model
    )

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)